In [ ]:
import numpy as np

# heisenberg model
class HB1DNN:

    def __init__(self, sites=None):
        self.model_flag = "hb"                            # set an explanation flag
        self.comment = "HB1DNN, H= Jx*X_i X_{i+1}\
                         + Jy*Y_i Y_{i+1}\
                         + Jz*Z_i Z_{i+1}\
                         + hx*X_i + hy*Y_i + hz*Z_i"      # a free comment
        self.params = {"Jx":None,                         # magnetic coupling X
                    "Jy":None,                            # magnetic coupling Y
                    "Jz":None,                            # magnetic coupling Z
                    "hx":None,                            # magnetic field X
                    "hy":None,                            # magnetic field Y
                    "hz":None}                            # magnetic field Z
        if type(sites)!=tuple:
            self.sites = (sites,)                         # make sites to tuple
        else:
            self.sites = sites                            # number of sites
        self.tot_sites = np.prod(sites)                   # total number of sites
        self.ldim = 2                                     # dimension of local hilbert space
        self.channels = 3                                 # number of channels
        self.mdim = self.channels+2                       # mpo dimension
        self.coordination_physical = (1,)                 # physical coordination number
        self.coordination_virtual = (2,)                  # virtual coordination number
        self.lattice = "1D"                               # lattice structure
        self.geometry = "chain"                           # system geometry chain
        self.snake = "linear"                             # snaking


In [ ]:
""" Ground State Calculations for: Bond-Alternating Heisenberg Model, chain """

import sys
sys.path.append("/home/mmarahrens/tensor-network-bundle")

import numpy as np
import datetime as dt

from tnb.algorithm.idmrg import iDMRG
from tnb.model.heisenberg.hb import HB1DNN

if __name__=="__main__":

    L = 4                                                    # define the number of sites
    bhm = HB1DNN(sites=L)                                    # define heisenberg object
    Ja = 1.0
    Jb_list = list(np.linspace(0.1, 0.9, 9)) +\
              list(np.linspace(1.1, 2.0, 10))                # parameters for simulation

    # Flag for date and time
    dt_flag = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

    # Simulation
    with open("data/idmrg" + dt_flag + ".p", "wb") as file:
        sim = iDMRG(model = bhm)                             # initilize idmrg simulation object
        sim.init_iterables()                                 # initilize iterable list for idmrg
        sim.chi_max = 200                                    # set max bond dimenson
        sim.N_max = 100                                      # set max simulation steps
        sim.disc_weight = 1e-12                              # set the discarded weight

        # Data storage
        sim.data.datafile = file                             # set the data file
        for Jb in Jb_list:
            Jb = round(Jb, 2)
            print(Jb)
            bhm.params = {"Ja":Ja, "Jb":Jb}
            sim.init_mps()                                   # initilize/reset mps
            # prepare parameters
            J_list = []; h_list = []
            for i in range(L):
                if i%2==0:
                    J_list.append((Ja,Ja,Ja))
                if i%2==1:
                    J_list.append((Jb,Jb,Jb))
                h_list.append((0.0,0.0,0.0))
            sim.init_mpo(J = J_list, h = h_list)             # initilize/reset matrix product operator
            sim.init_env()                                   # initilize/reset environments
            sim.run()                                        # run the simulation
            sim.data.dump_data(sim.collect_data())           # collect data and dump it to file as dictionary


In [ ]:
""" Ground State Calculations for: Bond-Alternating Heisenberg Model, chain """

import sys
sys.path.append("/home/mmarahrens/tensor-network-bundle")

import pickle
import numpy as np
import datetime as dt

from tnb.algorithm.itebd import iTEBD   # time evolution class
from tnb.operator.mpo import Mpo        # mpo class

def loadfile(filename):
    data = []
    with open(filename, "rb") as f:
        while True:
            try:
                #yield pickle.load(f)
                data.append(pickle.load(f))
            except EOFError:
                break
    return data

if __name__=="__main__":
    
    # temporary load data idmrg20181029_150307.p
    data = loadfile("/home/mmarahrens/Dropbox/data_mirco/bah/idmrg20181029_150307.p")
    
    # get mps, mpo and system instances
    for i in data:
        Ja = i["model"].params["Ja"]
        Jb = i["model"].params["Jb"]
        if round(Jb,2) == 0.5:
            mps = i["mps"]
            mpo = i["mpo"]
            system = i["system"]
            break      
     # "close" the data
    del data

    # prepare quench parameters
    L = system.sites
    Jaq = 1.0; Jbq = 0.5
    J_list = []; h_list = []
    for i in range(L):
        if i%2==0:
            J_list.append((Jaq, Jaq, Jaq))
        if i%2==1:
            J_list.append((Jbq, Jbq, Jbq))
        h_list.append((0.0, 0.0, 0.0))

    # prepare quench mpo
    mpo_quench = Mpo(local_dim=2, mpo_dim=5, sites=L, model_flag="hb")
    mpo_quench.init_mpo(params={"J":J_list, "h":h_list})
    
    # prepare the simulation
    sim = iTEBD(mps = mps, mpo = mpo_quench, system = system)   # initilize itebd simulation object
    sim.init_unitaries_from_mpo()                               # initilize unitaries from mpo
    sim.chi_max = 2000                                          # set max bond dimenson
    sim.N_max = 100                                             # set max simulation steps
    sim.disc_weight = 1e-12                                     # set the discarded weight
    
    iterable_list = system.iterable_list
    
    #mps.orthogonalization(system.iterable_list)
    
    A_list = mps.A_list[:]
    l_list = mps.l_list[:]
    B_list = mps.B_list[:]
    M_list = mps.M_list[:]
    
    # run the simulation
    #sim.run()
        
    

In [ ]:
""" Ground State Calculations for: Bond-Alternating Heisenberg Model, chain """

import sys
sys.path.append("/home/mmarahrens/tensor-network-bundle")

import pickle
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import pandas as pd

def loadfile(filename):
    data = []
    with open(filename, "rb") as f:
        while True:
            try:
                #yield pickle.load(f)
                data.append(pickle.load(f))
            except EOFError:
                break
    return data

if __name__=="__main__":
    
    # load data
    data = loadfile("data/phasediagram20181023_154125.p")
    
    # list for simulation data frames
    simframes = []

In [ ]:
""" Ground State Calculations for: Bond-Alternating Heisenberg Model, chain """

import sys
sys.path.append("/home/mmarahrens/tensor-network-bundle")

import pickle
import numpy as np
import datetime as dt

from tnb.algorithm.itebd import iTEBD   # time evolution class
from tnb.operator.mpo import Mpo        # mpo class

def loadfile(filename):
    data = []
    with open(filename, "rb") as f:
        while True:
            try:
                #yield pickle.load(f)
                data.append(pickle.load(f))
            except EOFError:
                break
    return data

if __name__=="__main__":
    
    # temporary load data idmrg20181029_150307.p
    data = loadfile("/home/mmarahrens/Dropbox/data_mirco/bah/idmrg20181029_150307.p")
    
    # get mps, mpo and system instances
    for i in data:
        Ja = i["model"].params["Ja"]
        Jb = i["model"].params["Jb"]
        if round(Jb,2) == 0.5:
            mps = i["mps"]
            mpo = i["mpo"]
            system = i["system"]
            break      
     # "close" the data
    del data

    # prepare quench parameters
    L = system.sites
    Jaq = 1.0; Jbq = 0.5
    J_list = []; h_list = []
    for i in range(L):
        if i%2==0:
            J_list.append((Jaq, Jaq, Jaq))
        if i%2==1:
            J_list.append((Jbq, Jbq, Jbq))
        h_list.append((0.0, 0.0, 0.0))

    # prepare quench mpo
    mpo_quench = Mpo(local_dim=2, mpo_dim=5, sites=L, model_flag="hb")
    mpo_quench.init_mpo(params={"J":J_list, "h":h_list})
    
    # prepare the simulation
    sim = iTEBD(mps = mps, mpo = mpo_quench, system = system)   # initilize itebd simulation object
    sim.init_unitaries_from_mpo()                               # initilize unitaries from mpo
    sim.chi_max = 2000                                          # set max bond dimenson
    sim.N_max = 100                                             # set max simulation steps
    sim.disc_weight = 1e-12                                     # set the discarded weight
    
    iterable_list = system.iterable_list
    
    #mps.orthogonalization(system.iterable_list)
    
    A_list = mps.A_list[:]
    l_list = mps.l_list[:]
    B_list = mps.B_list[:]
    M_list = mps.M_list[:]
    
    # run the simulation
    #sim.run()
        
    